# Resetting formatting in `unichart`

Every formatting change you make in a `UnichartNotebook` is **stateful** — it
sticks until you clear it. This notebook is a tour of *every* way to undo
formatting, from a single attribute back up to a full wipe.

Two rules cover almost everything:

1. **Any formatting setter accepts `'reset'` as its value** — restoring the
   default for whatever that call targets (a dataset attribute, a variable
   override, or a decoration).
2. **`reset_format()` is the single bulk-reset hub** — with optional *scopes*
   (`'sets'`, `'vars'`, `'lines'`, `'highlights'`, `'scales'`, `'fonts'`,
   `'plot_size'`, `'defaults'`, `'all'`) and dataset selectors.

We'll build up a heavily-formatted figure, then peel the formatting back off it
piece by piece.

## Setup — data and a baseline plot

We synthesize a few "test runs" that share a schema (`time`, `temperature`,
`pressure`), load them as one dataset per run, and draw a plain plot.

In [1]:
# --- make repo-root importable (notebook lives in demo_notebooks/) ---
import sys, os
_repo_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import numpy as np
import pandas as pd
from unichart import UnichartNotebook

rng = np.random.default_rng(0)
frames = []
for run in range(4):
    t = np.linspace(0, 100, 60)
    frames.append(pd.DataFrame({
        'run_id':      run,
        'run_name':    f'run {run}',
        'time':        t,
        'temperature': 60 + 8 * run + 15 * np.sin(t / 12) + rng.normal(0, 1.2, t.size),
        'pressure':    30 + 3 * run + 5 * np.cos(t / 18) + rng.normal(0, 0.6, t.size),
    }))
df = pd.concat(frames, ignore_index=True)

nb = UnichartNotebook()
nb.load_df(df, set_idx_column='run_id', set_name_column='run_name')
nb.plot(x='time', y=['temperature', 'pressure'])

UniChart Notebook Environment Initialized.
Loaded Set 0: run 0
Loaded Set 1: run 1
Loaded Set 2: run 2
Loaded Set 3: run 3


### Apply a pile of formatting

Now we deliberately override *everything* so each reset below has something to
undo: per-dataset styles, per-variable overrides, reference lines, a highlight
band, fixed axis scales, fonts, and a pinned plot size.

In [15]:
# per-dataset styles
nb.color(0, 'red')
nb.color(1, 'green')
nb.marker('all', 's')
nb.linestyle([0, 1], '--')
nb.linewidth(0, 4)
nb.markersize(2, 12)
nb.alpha('all', 0.6)

# per-variable overrides (win over per-dataset style wherever that column plots)
nb.var_format('temperature', linestyle=':', marker='^')
nb.var_format('pressure', color='purple')

# decorations
nb.line('temperature', level=80, color='black', linestyle='--')   # horizontal ref line
nb.line('time', level=50, color='gray')                            # vertical ref line
nb.highlight('time', (20, 40), color='yellow', alpha=0.2)          # shaded band
nb.scale('temperature', (40, 110))                                 # fixed y-range

# appearance
nb.set_font_sizes(suptitle=22, legend=16)
nb.set_plot_size(width=12, height=8)

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Fully formatted')

Limits set for 'temperature': (40, 110)


## Rule 1 — `'reset'` as a value (per-attribute undo)

Every setter that takes a value also accepts the literal string `'reset'`. This
is the finest-grained undo: one attribute, on one target.

### 1a. Per-dataset attributes

`'reset'` restores that attribute to what the dataset would get from the
notebook's maps / defaults (`color_map`, `marker_map`, `set_default_format`).

In [16]:
nb.color(0, 'reset')          # dataset 0 → back to its color_map color
nb.marker('all', 'reset')     # every dataset → back to its marker_map marker
nb.linestyle([0, 1], 'reset') # drop the dashed override on 0 and 1
nb.linewidth(0, 'reset')      # dataset 0 → default line width
nb.markersize(2, 'reset')     # dataset 2 → default marker size
nb.alpha('all', 'reset')      # every dataset → default opacity

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Dataset styles reset')

The same `'reset'` value works for **every** per-dataset setter, not just the
six shown above — `nb.fill(0, 'reset')`, `nb.edgewidth(0, 'reset')`, and
`nb.hue(0, 'reset')` all clear their attribute the same way.

The selector is flexible: an int, a list of ints, `'all'`, `'selected'`, or a
`Dataset`. So `nb.color('selected', 'reset')` clears the color only on the
currently-selected sets.

### 1b. Per-variable overrides

The same setters, when handed a **column name** instead of a dataset selector,
target the per-variable override layer. `'reset'` there drops the override so
the variable falls back to per-dataset style.

In [17]:
# these two forms are equivalent — both clear the *variable* override:
nb.color('pressure', 'reset')                 # via the color() setter
nb.var_format('temperature', linestyle='reset')  # per-attribute on var_format

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Some var overrides reset')

`var_format` resets **per attribute** — passing `'reset'` to just one keyword
clears only that override and leaves the others (e.g. `temperature` still keeps
its `marker='^'` above). To clear *all* overrides on a variable in one shot, use
`clear_var_format`:

In [28]:
nb.clear_var_format?

Signature: nb.clear_var_format(variable=None)
Docstring:
Clear variable formatting. Pass None (or no arg) to clear everything.
Equivalent to ``reset_format(vars=variable)`` / ``reset_format('vars')``.
File:      ~/Documents/GitHub/unichart/unichart.py
Type:      method

In [18]:
nb.clear_var_format('temperature')   # drop every override on 'temperature'
# nb.clear_var_format()              # (no arg) → drop overrides on ALL variables

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='All temperature overrides cleared')

### 1c. Decorations — lines, highlights, scales

Reference lines, highlight bands, and fixed axis ranges reset through their own
setters. Pass `'reset'` (alias `'clear'`) as the *value* argument, and use
`column='all'` to clear that decoration everywhere.

In [19]:
# lines — remove the ones keyed to a single column, or all of them
nb.line('temperature', 'reset')   # drop the horizontal line on temperature
nb.line('all', 'reset')           # remove every reference line ('clear' also works)

# highlights — same pattern (range_tuple = 'reset')
nb.highlight('time', 'reset')     # remove highlights keyed to 'time'
# nb.highlight('all', 'reset')    # remove every highlight

# scales / axis limits — range_tuple = 'reset' (or None)
nb.scale('temperature', 'reset')  # release the fixed temperature range
nb.scale('all', 'reset')          # clear every fixed axis range

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Decorations cleared')

Limits cleared for 'temperature'.
All axis limits cleared.


## Rule 2 — `reset_format()`, the bulk-reset hub

`reset_format()` is the one entry point for wiping formatting in bulk. Its
behavior is driven by *scopes* and *dataset selectors*.

First, re-apply a batch of formatting so there's something to sweep:

In [20]:
nb.color('all', 'orange')
nb.marker('all', 'D')
nb.var_format('pressure', color='navy')
nb.line('temperature', level=75)
nb.highlight('time', (10, 30))
nb.scale('pressure', (20, 55))
nb.set_font_sizes(all=18)
nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Re-formatted for the bulk demo')

Limits set for 'pressure': (20, 55)


### 2a. No arguments — reset all *applied* formatting

A bare `reset_format()` sweeps every scope of applied formatting: dataset
styles, variable overrides, lines, highlights, axis limits, fonts, and the
pinned plot size. It **deliberately keeps** your `set_default_format` state —
resetting applied formatting shouldn't silently discard the defaults you chose.
(The print-out lists exactly what it touched.)

In [21]:
nb.reset_format()
nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Everything applied, reset')

Reset: dataset formatting, variable formats, lines, highlights, axis limits, font sizes, plot size.


### 2b. Named scopes — reset only some layers

Pass one or more scope names. Handy aliases work too (`'scale'`→`'scales'`,
`'var'`/`'variable'`→`'vars'`, `'line'`→`'lines'`, `'font'`→`'fonts'`, …).

In [22]:
nb.color('all', 'teal'); nb.line('temperature', level=70)
nb.highlight('time', (10, 30)); nb.scale('pressure', (20, 55))

nb.reset_format('lines', 'scales')   # clear ONLY reference lines and axis limits
# color + highlight survive; lines + scales are gone
nb.plot(x='time', y=['temperature', 'pressure'], suptitle="reset_format('lines', 'scales')")

Limits set for 'pressure': (20, 55)
Reset: lines, axis limits.


### 2c. Dataset selectors — reset only some sets

A non-string positional (int, list, or `Dataset`) is read as a dataset
selector: it resets *only those datasets'* styles and leaves every other scope
untouched. The explicit form is `reset_format('sets', uset_slice=...)`.

In [23]:
nb.color('all', 'crimson'); nb.marker('all', 'x')

nb.reset_format([0, 1])              # reset styles on datasets 0 and 1 only
# nb.reset_format('sets', uset_slice=0)   # explicit equivalent for one set
nb.plot(x='time', y=['temperature', 'pressure'], suptitle='reset_format([0, 1]) — sets 2,3 keep style')

Reset: dataset formatting (2 sets).


### 2d. Targeted keyword resets — one variable / line / highlight / scale

The `vars`, `lines`, `highlights`, and `scale` keywords accept a **name** (or
list of names) to narrow *which entries* within a scope get cleared. To clear
just those entries **and nothing else**, name the scope positionally as well:

> ⚠️ **Gotcha.** A targeted keyword narrows a scope but does **not**, on its own,
> restrict *which scopes run*. A bare `reset_format(vars='temperature')` still
> performs the full default sweep (datasets, lines, highlights, scales, fonts,
> plot size) — it just limits the *vars* part to `temperature`. Pair the keyword
> with its scope name — `reset_format('vars', vars='temperature')` — to touch
> only that one thing. (The `Reset:` print-out always tells you exactly what was
> swept, so you can confirm.)

In [24]:
nb.var_format('temperature', color='red'); nb.var_format('pressure', color='navy')
nb.line('temperature', level=75); nb.line('pressure', level=45)

nb.reset_format('vars', vars='temperature')   # clear ONLY 'temperature' overrides
nb.reset_format('lines', lines='pressure')    # remove ONLY the 'pressure' line
# 'pressure' override and 'temperature' line both survive.
nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Targeted var/line resets')

Reset: variable formats (temperature).
Reset: lines (pressure).


### 2e. Boolean include / exclude

The same keywords take `True` / `False` to add or subtract a scope from the
sweep — e.g. "reset everything *except* the datasets".

In [25]:
nb.color('all', 'darkgreen'); nb.line('temperature', level=70); nb.scale('pressure', (20, 55))

nb.reset_format(sets=False)   # sweep everything applied EXCEPT dataset styles
# datasets stay green; the line and the scale are cleared
nb.plot(x='time', y=['temperature', 'pressure'], suptitle='reset_format(sets=False)')

Limits set for 'pressure': (20, 55)
Reset: variable formats, lines, highlights, axis limits, font sizes, plot size.


### 2f. `'all'` — applied formatting *and* the stored defaults

`reset_format('all')` is the only bulk call that also wipes `set_default_format`
state (per-dataset default styles, `figsize`, per-call plot defaults). Use it
for a truly clean slate.

In [26]:
nb.set_default_format(markersize=10, linestyle='--', figsize=(9, 5))
nb.color('all', 'red')

nb.reset_format('all')   # applied formatting + set_default_format state → built-ins
nb.plot(x='time', y=['temperature', 'pressure'], suptitle="reset_format('all') — clean slate")

Reset: style/plot defaults, dataset formatting, variable formats, lines, highlights, axis limits, font sizes, plot size.


## Dedicated reset methods (equivalents)

A few methods carry their own reset switch. Each is exactly equivalent to the
matching `reset_format` scope — use whichever reads better at the call site.

| Dedicated call | Equivalent scope |
|---|---|
| `set_default_format(reset=True)` | `reset_format('defaults')` |
| `set_font_sizes(reset=True)` | `reset_format('fonts')` |
| `set_plot_size(reset=True)` | `reset_format('plot_size')` |
| `clear_var_format()` | `reset_format('vars')` |
| `line('all', 'reset')` | `reset_format('lines')` |
| `highlight('all', 'reset')` | `reset_format('highlights')` |
| `scale('all', 'reset')` | `reset_format('scales')` |

In [27]:
nb.set_default_format(markersize=9, figsize=(9, 5))
nb.set_font_sizes(suptitle=24)
nb.set_plot_size(width=500, height=320)
nb.var_format('pressure', color='navy')

nb.set_default_format(reset=True)   # == reset_format('defaults')
nb.set_font_sizes(reset=True)       # == reset_format('fonts')
nb.set_plot_size(reset=True)        # == reset_format('plot_size')
nb.clear_var_format()               # == reset_format('vars')

nb.plot(x='time', y=['temperature', 'pressure'], suptitle='Dedicated resets — back to defaults')

Default format, figsize, and plot defaults reset to built-ins.


## Recap

**Rule 1 — per-attribute `'reset'`** (finest grain):

```python
nb.color(0, 'reset')                  # one dataset attribute
nb.marker('all', 'reset')             # an attribute across all sets
nb.color('pressure', 'reset')         # one variable override
nb.var_format('temp', color='reset')  # one attribute of one variable
nb.line('all', 'reset')               # a decoration ('clear' also works)
nb.highlight('time', 'reset')
nb.scale('all', 'reset')
```

**Rule 2 — `reset_format()`** (bulk hub):

```python
nb.reset_format()                    # all APPLIED formatting (keeps defaults)
nb.reset_format('lines', 'scales')       # only named scopes
nb.reset_format([0, 1])                  # only these datasets
nb.reset_format('vars', vars='temp')     # ONLY that variable's overrides
nb.reset_format(sets=False)              # everything except datasets
nb.reset_format('all')                   # applied formatting + stored defaults
```

> A bare `reset_format(vars='temp')` (no `'vars'` scope name) still sweeps
> everything applied — it only *narrows* the vars part. Name the scope, or use
> `clear_var_format('temp')`, to touch just the one variable.

**Dedicated equivalents:** `set_default_format(reset=True)`,
`set_font_sizes(reset=True)`, `set_plot_size(reset=True)`,
`clear_var_format()`.

Rule of thumb: reach for **`'reset'`** to undo one thing, and **`reset_format`**
to undo many. When in doubt, `nb.reset_format()` gets you back to a plain plot
without discarding the defaults you configured.